# 🌳 Day 01a: B-Tree Indexing — How Indexes Make Queries Fast

---

## 🎯 What You'll Master Today
- B-Tree vs B+ Tree structure
- Clustered vs non-clustered indexes
- Composite indexes and leftmost prefix rule
- When indexes help vs hurt

---

## 🏗️ The Library Card Catalog Analogy

> Without an index: read every page of every book to find one fact (full table scan).
> With an index: look up the card catalog (B+ Tree) → get shelf location → jump directly.

```
╔══════════════════════════════════════════════════════════════════╗
║             B+ TREE INDEX STRUCTURE                              ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║              [  30  |  60  ]         ← Root (internal)          ║
║             /       |       \                                    ║
║     [10|20]      [40|50]     [70|80]  ← Internal nodes          ║
║     / | \        / | \       / | \                              ║
║  [5,10] [20,25] [30,40] [50,55] [60,70] [80,90]  ← Leaf nodes ║
║    →      →       →       →       →      (linked list)         ║
║                                                                  ║
║  B+ Tree Properties:                                            ║
║  • Data only in leaf nodes (internal = keys + pointers)         ║
║  • Leaves linked → efficient range scans                       ║
║  • Balanced: all leaves at same depth                           ║
║  • O(log n) search, insert, delete                              ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================
# 1. B+ Tree Simulation
# ============================================

class BPlusTreeNode:
    def __init__(self, is_leaf=False, order=4):
        self.keys = []
        self.children = []
        self.is_leaf = is_leaf
        self.next_leaf = None  # linked list for range scans
        self.order = order

class SimpleBPlusTree:
    """Simplified B+ Tree for demonstration."""
    def __init__(self, order=4):
        self.root = BPlusTreeNode(is_leaf=True, order=order)
        self.order = order
        self.io_count = 0  # Track disk I/O (page reads)
    
    def search(self, key):
        """Search for key, tracking I/O."""
        self.io_count = 0
        node = self.root
        path = []
        
        while not node.is_leaf:
            self.io_count += 1
            path.append(f"Internal{node.keys}")
            i = 0
            while i < len(node.keys) and key >= node.keys[i]:
                i += 1
            node = node.children[i]
        
        self.io_count += 1
        path.append(f"Leaf{node.keys}")
        found = key in node.keys
        return found, self.io_count, path
    
    def insert_simple(self, key):
        """Simplified insert (no splitting for demo)."""
        node = self.root
        while not node.is_leaf:
            i = 0
            while i < len(node.keys) and key >= node.keys[i]:
                i += 1
            node = node.children[i]
        node.keys.append(key)
        node.keys.sort()

# Build a tree manually for demo
tree = SimpleBPlusTree(order=4)
root = BPlusTreeNode(is_leaf=False)
root.keys = [30, 60]

left = BPlusTreeNode(is_leaf=True)
left.keys = [10, 20, 25]
mid = BPlusTreeNode(is_leaf=True)
mid.keys = [30, 40, 50]
right = BPlusTreeNode(is_leaf=True)
right.keys = [60, 70, 80]

left.next_leaf = mid
mid.next_leaf = right
root.children = [left, mid, right]
tree.root = root

# Search demo
for key in [40, 10, 80, 55]:
    found, ios, path = tree.search(key)
    status = "FOUND" if found else "NOT FOUND"
    print(f"  Search({key}): {status} | I/Os: {ios} | Path: {' → '.join(path)}")

print(f"\n  💡 1 million rows, order 100: only ~3 I/Os to find any row!")
print(f"  Full scan: 1,000,000 I/Os. B+ Tree: ~log₁₀₀(1M) ≈ 3 I/Os")

In [ ]:
# ============================================
# 2. Clustered vs Non-Clustered Index
# ============================================

print("""
  ╔════════════════════════════════════════════════════════════════╗
  ║        CLUSTERED vs NON-CLUSTERED INDEX                       ║
  ╠════════════════════════════════════════════════════════════════╣
  ║                                                                ║
  ║  CLUSTERED (1 per table):                                     ║
  ║  ┌──────────────┐                                              ║
  ║  │  B+ Tree      │  Leaf nodes = actual data rows             ║
  ║  │  Index + Data  │  Data physically sorted by index key      ║
  ║  └──────────────┘  (like a phone book sorted by name)         ║
  ║                                                                ║
  ║  NON-CLUSTERED (many per table):                               ║
  ║  ┌──────────────┐                                              ║
  ║  │  B+ Tree      │  Leaf nodes = pointers to data rows        ║
  ║  │  Index → Ptr   │  Extra lookup needed (bookmark lookup)     ║
  ║  └──────────────┘  (like book index → page numbers)           ║
  ║                                                                ║
  ║  MySQL InnoDB:                                                 ║
  ║  • Primary Key = clustered index (always!)                    ║  
  ║  • Secondary indexes store PK value (not row pointer)         ║
  ║  • Secondary lookup: index → PK → clustered index → row      ║
  ║                                                                ║
  ╚════════════════════════════════════════════════════════════════╝
""")

# Simulate clustered vs non-clustered
print("  Clustered Index (data sorted by key):")
clustered = sorted([
    {"id": 3, "name": "Charlie", "age": 30},
    {"id": 1, "name": "Alice",   "age": 25},
    {"id": 2, "name": "Bob",     "age": 28},
], key=lambda x: x["id"])
for row in clustered:
    print(f"    Page: id={row['id']} | {row['name']} | {row['age']}")

print("\n  Non-Clustered Index on 'name':")
nc_index = sorted([(r["name"], r["id"]) for r in clustered])
for name, pk in nc_index:
    print(f"    Index: name='{name}' → PK={pk} → (lookup clustered index)")

In [ ]:
# ============================================
# 3. Composite Index & Leftmost Prefix Rule
# ============================================

print("""
  Composite Index: CREATE INDEX idx ON users(country, city, zipcode);
  
  ┌───────────────────────────────────────────────────────────────────┐
  │ Query                                │ Uses Index? │ Why?         │
  ├──────────────────────────────────────┼─────────────┼──────────────┤
  │ WHERE country='US'                   │  ✓ Yes      │ Leftmost col│
  │ WHERE country='US' AND city='NYC'    │  ✓ Yes      │ First 2 cols│
  │ WHERE country='US' AND city='NYC'    │  ✓ Yes      │ All 3 cols  │
  │   AND zipcode='10001'                │             │              │
  │ WHERE city='NYC'                     │  ✗ No!      │ Skips first │
  │ WHERE zipcode='10001'                │  ✗ No!      │ Skips first │
  │ WHERE country='US' AND zipcode='100' │  ◐ Partial  │ Skips city  │
  └──────────────────────────────────────┴─────────────┴──────────────┘
  
  💡 LEFTMOST PREFIX RULE: Index (A,B,C) supports:
     (A), (A,B), (A,B,C) — NOT (B), (C), (B,C)
""")

# Simulate index usage
users = [
    {"country": "US", "city": "NYC", "zip": "10001", "name": "Alice"},
    {"country": "US", "city": "NYC", "zip": "10002", "name": "Bob"},
    {"country": "US", "city": "LA",  "zip": "90001", "name": "Charlie"},
    {"country": "UK", "city": "LON", "zip": "SW1",   "name": "Diana"},
]

# Index scan (uses leftmost prefix)
print("  Index scan: WHERE country='US' AND city='NYC'")
result = [u for u in users if u["country"] == "US" and u["city"] == "NYC"]
for r in result:
    print(f"    → {r['name']} ({r['zip']})")

In [ ]:
# ============================================
# 4. When Indexes Help vs Hurt
# ============================================

import math

print("""
  ┌───────────────────────────────────────────────────────────────┐
  │ INDEX HELPS ✓                  │ INDEX HURTS ✗               │
  ├───────────────────────────────┼─────────────────────────────┤
  │ WHERE equality/range          │ Small tables (< 1000 rows) │
  │ JOIN conditions               │ Low selectivity columns    │
  │ ORDER BY / GROUP BY           │ (gender: only M/F)         │
  │ Covering index (all cols)     │ Heavy INSERT/UPDATE tables │
  │ High selectivity (unique)     │ Columns rarely queried      │
  │ Large tables                  │ Expressions (WHERE f(col))  │
  └───────────────────────────────┴─────────────────────────────┘
""")

# Cost comparison
rows = 1_000_000
order = 100  # typical B+ tree fanout

full_scan = rows
index_scan = math.ceil(math.log(rows, order)) + 1  # tree height + data page

print(f"  For {rows:,} rows (fanout={order}):")
print(f"    Full table scan: ~{full_scan:,} page reads")
print(f"    B+ Tree lookup:  ~{index_scan} page reads")
print(f"    Speedup: {full_scan // index_scan:,}x faster!")
print(f"\n  But INDEX adds overhead:")
print(f"    INSERT: +{index_scan} I/Os per index (find + update leaf)")
print(f"    Storage: ~10-30% extra space per index")

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | B-Tree vs B+ Tree? | B+ Tree: data only in leaves + leaves linked. Better for range scans |
| 2 | Clustered vs non-clustered? | Clustered: data sorted by index (1 per table). Non-clustered: pointer to data |
| 3 | Leftmost prefix rule? | Composite index (A,B,C) only usable for queries starting with A |
| 4 | When NOT to use index? | Small tables, low selectivity, heavy writes, expressions in WHERE |
| 5 | What is a covering index? | Index contains ALL columns needed by query → no table lookup needed |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • B+ Tree: balanced, O(log n), leaves linked           │
## │  • Clustered = data in leaf. Non-clustered = pointer     │
## │  • Composite: leftmost prefix rule governs usage         │
## │  • Covering index: avoids table lookup (fastest!)        │
## │  • Indexes cost: write overhead + storage                │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Hash Indexing** — Hash tables, bitmap indexes, comparison